In [1]:
from pyspark.sql import functions as F

# 1. Silver transform
BRONZE = "checkin_bronze"
SILVER = "checkin_silver"

df_checkin = spark.table(BRONZE)

StatementMeta(, 102c6955-0fdb-4887-8c4c-3b94ebc6db05, 3, Finished, Available, Finished, False)

In [5]:
base_cols = [
    "business_id",
    "date",
    "_ingest_ts",
    "_source_file",
    "_batch_id"
]

df_checkin_silver = (
    df_checkin
    .select(*base_cols)
    .filter(F.col("business_id").isNotNull())
    .filter(F.col("date").isNotNull())
    .dropDuplicates(["business_id"])
)

StatementMeta(, 102c6955-0fdb-4887-8c4c-3b94ebc6db05, 7, Finished, Available, Finished, False)

In [6]:
(
    df_checkin_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER)

)

StatementMeta(, 102c6955-0fdb-4887-8c4c-3b94ebc6db05, 8, Finished, Available, Finished, False)

In [7]:
print("checkin_silver rows = ", spark.table("checkin_silver").count())

StatementMeta(, 102c6955-0fdb-4887-8c4c-3b94ebc6db05, 9, Finished, Available, Finished, False)

checkin_silver rows =  131930


In [ ]:
mssparkutils.notebook.exit("SUCCESS")

In [8]:
#2. Data Quality Check
dup = (
    df_checkin_silver_DQ
    .groupBy("business_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("duplicate business_id =", dup)

df_checkin_silver_DQ.select(
    F.sum(F.col("business_id").isNull().cast("int")).alias("null_business_id"),
    F.sum(F.col("date").isNull().cast("int")).alias("null_date")
).show()

StatementMeta(, 102c6955-0fdb-4887-8c4c-3b94ebc6db05, 10, Finished, Available, Finished, False)

checkin_silver rows = 131930
duplicate business_id = 0
+----------------+---------+
|null_business_id|null_date|
+----------------+---------+
|               0|        0|
+----------------+---------+

